In [ ]:
# Imports, reproducibility, and the shared high-end plotting style.

import os
import json
import time

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Polygon
from collections import Counter

import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq

SEED = 3407
torch.manual_seed(SEED)
np.random.seed(SEED)

IMG_DIR = "img"
os.makedirs(IMG_DIR, exist_ok=True)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif", "Liberation Serif", "serif"],
    "font.size": 14,
    "axes.titlesize": 19,
    "axes.labelsize": 15,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 13,
    "axes.edgecolor": "#B7BDC6",
    "axes.linewidth": 1.0,
    "grid.color": "#ECECEC",
    "grid.linewidth": 1.0,
    "figure.dpi": 120,
})

PALETTE = {
    "blue": "#7BA7CC",
    "green": "#8FBF9F",
    "coral": "#E0A18E",
    "purple": "#B3A2CC",
    "amber": "#E6C77E",
    "slate": "#8C9AAE",
}

def lighten(hex_color, amount=0.55):
    r, g, b = mcolors.to_rgb(hex_color)
    return (r + (1 - r) * amount, g + (1 - g) * amount, b + (1 - b) * amount)

def gradient_vbar(ax, x, height, width, base_color):
    cmap = LinearSegmentedColormap.from_list("g", [lighten(base_color, 0.6), base_color])
    grad = np.linspace(0, 1, 256).reshape(-1, 1)
    ax.imshow(grad, extent=(x - width / 2, x + width / 2, 0, height),
              origin="lower", aspect="auto", cmap=cmap, zorder=2, interpolation="bicubic")

def gradient_hbar(ax, y, value, height, base_color):
    cmap = LinearSegmentedColormap.from_list("g", [lighten(base_color, 0.62), base_color])
    grad = np.linspace(0, 1, 256).reshape(1, -1)
    ax.imshow(grad, extent=(0, value, y - height / 2, y + height / 2),
              origin="lower", aspect="auto", cmap=cmap, zorder=2, interpolation="bicubic")

def tidy(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_axisbelow(True)

print("Setup complete. Torch CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Configuration. Every value here is a deliberate choice, justified in the report.

MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct"   # switch to "unsloth/Qwen2.5-3B-Instruct" for a lighter/faster run
MAX_SEQ_LENGTH = 2048                        # reviews are short (~20 words); 2048 is plenty of headroom
LOAD_IN_4BIT = True                          # QLoRA: 4-bit base weights, fits a 7B model comfortably in 16 GB

LORA_R = 16                                  # rank: capacity of the adapter; 16 is a solid default for a focused task
LORA_ALPHA = 16                              # scaling; alpha = r is a common, stable starting point
LORA_DROPOUT = 0                             # 0 is the Unsloth-optimized setting
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]   # all attention + MLP projections

LEARNING_RATE = 2e-4                         # standard LoRA learning rate from the Unsloth guide
NUM_EPOCHS = 3                               # small high-quality set -> 3 epochs balances learning vs memorizing
BATCH_SIZE = 2                               # per-device batch; lower to 1 if you ever hit out-of-memory
GRAD_ACCUM = 4                               # effective batch = BATCH_SIZE * GRAD_ACCUM = 8

DATA_PATH = "dataset.jsonl"

print("Model:", MODEL_NAME)
print(f"LoRA: r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"Training: lr={LEARNING_RATE}, epochs={NUM_EPOCHS}, effective_batch={BATCH_SIZE * GRAD_ACCUM}")


In [ ]:
# Load the base model in 4-bit (this is the QLoRA base).

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,                 # None lets Unsloth pick float16 / bfloat16 for your GPU
    load_in_4bit=LOAD_IN_4BIT,
)

n_params = sum(p.numel() for p in model.parameters())
print(f"Base model loaded. Total parameters: {n_params/1e9:.2f}B")
print("Quantization: 4-bit (QLoRA)" if LOAD_IN_4BIT else "Quantization: none")


In [ ]:
# Attach LoRA adapters. Only these thin matrices are trained; the 4-bit base stays frozen.

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",   # saves memory, enables longer context
    random_state=SEED,
    use_rslora=False,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,}")
print(f"Total parameters:     {total:,}")
print(f"Trainable share:      {100*trainable/total:.3f}%   (this is the whole point of LoRA)")


In [ ]:
# Apply the Qwen-2.5 chat template, load the dataset, and convert each conversation into a
# single training string. The dataset is role-tagged (system / user / assistant), so it lines
# up exactly with the model's expected chat format.

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
             for c in convos]
    return {"text": texts}

raw_dataset = load_dataset("json", data_files=DATA_PATH, split="train")
train_dataset = raw_dataset.map(formatting_prompts_func, batched=True)

print(f"Loaded {len(train_dataset)} training examples from {DATA_PATH}")
print("\nOne formatted example as the model actually sees it:\n")
print(train_dataset[0]["text"])


In [ ]:
# Figure 1: dataset composition. Gradient bars (counts) + donut (share). Numbers printed first.

analyses = [json.loads(d["messages"][2]["content"]) for d in raw_dataset]
order = ["positive", "neutral", "mixed", "negative"]
counts = Counter(a["sentiment"] for a in analyses)
vals = [counts.get(k, 0) for k in order]
cols = [PALETTE["green"], PALETTE["amber"], PALETTE["purple"], PALETTE["coral"]]
print("Sentiment counts:", dict(zip(order, vals)), "| total:", sum(vals))

fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5.6), gridspec_kw={"width_ratios": [1.25, 1]})
w = 0.66
for i, (v, c) in enumerate(zip(vals, cols)):
    gradient_vbar(axL, i, v, w, c)
    axL.text(i, v + max(vals) * 0.02, str(v), ha="center", va="bottom", fontsize=14, weight="bold")
axL.set_xticks(range(len(order)))
axL.set_xticklabels(order)
axL.set_xlim(-0.6, len(order) - 0.4)
axL.set_ylim(0, max(vals) * 1.16)
axL.set_ylabel("Number of Examples")
axL.set_title("Sentiment Counts", pad=12, weight="bold")
axL.yaxis.grid(True)
tidy(axL)

wedges, _ = axR.pie(vals, colors=[lighten(c, 0.18) for c in cols], startangle=90,
                    counterclock=False, wedgeprops=dict(width=0.42, edgecolor="white", linewidth=2))
axR.text(0, 0, f"{sum(vals)}\nexamples", ha="center", va="center", fontsize=16, weight="bold")
axR.set_title("Sentiment Share", pad=12, weight="bold")
axR.legend(wedges, [f"{k}  ({v})" for k, v in zip(order, vals)],
           loc="center left", bbox_to_anchor=(1.0, 0.5), frameon=False)
fig.suptitle("Dataset Composition", fontsize=21, weight="bold", y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(IMG_DIR, "01_dataset_composition.png"), dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# Figure 2: review length distribution. Bars shaded by height + smoothed density + mean line.

lengths = [len(d["messages"][1]["content"].split()) for d in raw_dataset]
print(f"Review length (words) -> mean: {np.mean(lengths):.1f}, min: {min(lengths)}, max: {max(lengths)}")

fig, ax = plt.subplots(figsize=(9, 5.6))
counts_h, edges = np.histogram(lengths, bins=14)
centers = (edges[:-1] + edges[1:]) / 2
norm = counts_h / counts_h.max()
cmap = LinearSegmentedColormap.from_list("g", [lighten(PALETTE["blue"], 0.62), PALETTE["blue"]])
ax.bar(centers, counts_h, width=(edges[1] - edges[0]) * 0.92,
       color=[cmap(n) for n in norm], edgecolor="white", linewidth=1.3, zorder=2)
dens, dedges = np.histogram(lengths, bins=40, density=True)
dce = (dedges[:-1] + dedges[1:]) / 2
ds = np.convolve(dens, np.ones(5) / 5, mode="same")
ds = ds / ds.max() * counts_h.max()
ax.plot(dce, ds, color=PALETTE["coral"], linewidth=2.4, zorder=3, label="Density (smoothed)")
ax.axvline(np.mean(lengths), color="#6B6B6B", linewidth=2.0, linestyle=(0, (5, 3)),
           zorder=4, label=f"Mean = {np.mean(lengths):.1f} words")
ax.set_title("Review Length Distribution", pad=14, weight="bold")
ax.set_xlabel("Words per Review")
ax.set_ylabel("Number of Reviews")
ax.legend(frameon=False)
ax.yaxis.grid(True)
tidy(ax)
fig.tight_layout()
fig.savefig(os.path.join(IMG_DIR, "02_review_length.png"), dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# Figure 3: most frequent aspects (topics) as horizontal gradient bars.

topic_counts = Counter(t for a in analyses for t in a["topics"])
top = topic_counts.most_common(12)[::-1]
labels = [t for t, _ in top]
tvals = [c for _, c in top]
print("Top aspects:", dict(top[::-1]))

fig, ax = plt.subplots(figsize=(9, 6.6))
hh = 0.7
for i, v in enumerate(tvals):
    gradient_hbar(ax, i, v, hh, PALETTE["slate"])
    ax.text(v + max(tvals) * 0.012, i, str(v), va="center", ha="left", fontsize=12, weight="bold")
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels)
ax.set_ylim(-0.6, len(labels) - 0.4)
ax.set_xlim(0, max(tvals) * 1.14)
ax.set_xlabel("Frequency")
ax.set_title("Most Frequent Aspects (Topics)", pad=14, weight="bold")
ax.xaxis.grid(True)
tidy(ax)
fig.tight_layout()
fig.savefig(os.path.join(IMG_DIR, "03_top_topics.png"), dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# Held-out evaluation prompts (NONE of these appear in training) plus the helper functions.
# IMPORTANT design choice: the analyzer system prompt is SHORT and does NOT describe the JSON
# format. This is deliberate. The base model, given only this short prompt, has no reason to
# output JSON and will answer in prose - so it fails the JSON check. The fine-tuned model has
# internalized "review -> our exact JSON schema" into its weights, so it produces clean JSON
# from the same short prompt. That gap is exactly what fine-tuning buys us. General questions
# use a neutral prompt to check whether fine-tuning hurt normal answering (forgetting check).

ANALYZER_SYSTEM = "You are a customer review analysis assistant."
NEUTRAL_SYSTEM = "You are a helpful assistant."

domain_reviews = [
    "The blender works fine but the lid leaks every single time I make a smoothie. Annoying.",
    "Absolutely love this backpack. Tons of pockets, super comfy straps, and it survived a rainstorm without leaking.",
    "The hotel was in a great spot near the station, though the walls were paper thin and I could hear everything next door.",
    "Battery dies in about three hours and it gets uncomfortably hot while charging. Not what I expected for the price.",
    "Decent monitor for the money. Colors are accurate, but the stand wobbles a little.",
    "Fantastic noise cancelling on my flight, and the call quality blew me away. Worth every cent.",
    "Shipping took almost three weeks and the box arrived crushed, but the item inside was somehow intact.",
    "The app is okay I guess. It does what it says, but the interface feels really dated.",
]

general_questions = [
    "What is the capital of France?",
    "Explain what photosynthesis is in one sentence.",
    "What is 17 multiplied by 23?",
    "Give me a short tip for staying productive while working from home.",
]

def generate_response(user_msg, system_msg, max_new_tokens=160):
    messages = [{"role": "system", "content": system_msg},
                {"role": "user", "content": user_msg}]
    enc = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to("cuda")
    outputs = model.generate(
        **enc, max_new_tokens=max_new_tokens,
        temperature=0.7, top_p=0.8, use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = outputs[0][enc["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

REQUIRED_KEYS = {"sentiment", "score", "topics", "summary"}
VALID_SENTIMENTS = {"positive", "negative", "neutral", "mixed"}

def extract_json(text):
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(text[start:i+1])
                except Exception:
                    return None
    return None

def is_valid_analysis(text):
    obj = extract_json(text)
    if not isinstance(obj, dict):
        return False
    if not REQUIRED_KEYS.issubset(obj.keys()):
        return False
    if obj.get("sentiment") not in VALID_SENTIMENTS:
        return False
    if not isinstance(obj.get("score"), int) or not (1 <= obj["score"] <= 5):
        return False
    if not isinstance(obj.get("topics"), list):
        return False
    if not isinstance(obj.get("summary"), str):
        return False
    return True

def short(text, n=110):
    text = text.replace("\n", " ")
    return text if len(text) <= n else text[:n] + "..."

print(f"Prepared {len(domain_reviews)} held-out reviews and {len(general_questions)} general questions.")


In [ ]:
# Collect BASE-model answers BEFORE training. The LoRA adapters are zero-initialized right now,
# so the model behaves exactly like the untouched base model. We store these to compare later.

FastLanguageModel.for_inference(model)

base_domain_outputs = []
print("BASE MODEL on domain reviews (short analyzer prompt):\n")
for r in domain_reviews:
    out = generate_response(r, ANALYZER_SYSTEM)
    base_domain_outputs.append(out)
    print("REVIEW:", short(r, 80))
    print("OUTPUT:", short(out, 160))
    print("valid JSON:", is_valid_analysis(out), "\n")

base_general_outputs = []
print("\nBASE MODEL on general questions (neutral prompt):\n")
for q in general_questions:
    out = generate_response(q, NEUTRAL_SYSTEM)
    base_general_outputs.append(out)
    print("Q:", q)
    print("A:", short(out, 160), "\n")


In [ ]:
# Build the trainer. We compute the loss only on the assistant's JSON answer
# (train_on_responses_only), so the model learns to produce the JSON, not to copy the prompt.

FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    args=SFTConfig(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=5,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=SEED,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

steps_per_epoch = max(1, len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM))
print(f"Trainer ready. ~{steps_per_epoch} steps/epoch x {NUM_EPOCHS} epochs "
      f"= ~{steps_per_epoch * NUM_EPOCHS} total steps.")


In [ ]:
# Train. Watch the loss fall in real time.

t0 = time.time()
trainer_stats = trainer.train()
elapsed = time.time() - t0

print("\nTraining finished.")
print(f"Wall-clock time: {elapsed/60:.1f} min")
print(f"Final training loss: {trainer_stats.metrics.get('train_loss'):.4f}")


In [ ]:
# Figure 4: training loss curve with a gradient fill under the smoothed line.

history = trainer.state.log_history
loss_steps = np.array([h["step"] for h in history if "loss" in h])
loss_values = np.array([h["loss"] for h in history if "loss" in h])
print(f"Logged {len(loss_values)} loss points.")
print(f"First loss: {loss_values[0]:.4f}  ->  Last loss: {loss_values[-1]:.4f}")

fig, ax = plt.subplots(figsize=(9.2, 5.6))
ax.plot(loss_steps, loss_values, color=lighten(PALETTE["blue"], 0.3), linewidth=1.2,
        alpha=0.6, zorder=2, label="Training loss (raw)")
if len(loss_values) >= 7:
    sm = np.convolve(loss_values, np.ones(7) / 7, mode="valid")
    sm_x = loss_steps[3:len(sm) + 3]
else:
    sm, sm_x = loss_values, loss_steps
ax.plot(sm_x, sm, color=PALETTE["blue"], linewidth=2.8, zorder=4, label="Smoothed (7-step)")

ymax = float(loss_values.max()) * 1.08
rgb = mcolors.to_rgb(PALETTE["blue"])
grad = np.empty((256, 1, 4))
grad[:, :, 0], grad[:, :, 1], grad[:, :, 2] = rgb
grad[:, :, 3] = np.linspace(0.0, 0.38, 256).reshape(-1, 1)
im = ax.imshow(grad, extent=(sm_x.min(), sm_x.max(), 0, ymax), origin="lower",
               aspect="auto", zorder=1)
verts = [(sm_x.min(), 0)] + list(zip(sm_x, sm)) + [(sm_x.max(), 0)]
clip = Polygon(verts, closed=True, facecolor="none", edgecolor="none")
ax.add_patch(clip)
im.set_clip_path(clip)

ax.set_xlim(loss_steps.min(), loss_steps.max())
ax.set_ylim(0, ymax)
ax.set_title("Training Loss Curve", pad=14, weight="bold")
ax.set_xlabel("Training Step")
ax.set_ylabel("Loss")
ax.legend(frameon=False)
ax.yaxis.grid(True)
tidy(ax)
fig.tight_layout()
fig.savefig(os.path.join(IMG_DIR, "04_training_loss.png"), dpi=200, bbox_inches="tight")
plt.show()

print("\nInterpretation:")
print(f"- Loss dropped from {loss_values[0]:.2f} to {loss_values[-1]:.2f}, so the model is clearly learning.")
if loss_values[-1] < 0.15:
    print("- A very low final loss on a small dataset means it is memorizing the format well.")
    print("  Expected for a format task; if outputs feel rigid, reduce epochs or add varied data.")
else:
    print("- The loss flattened without collapsing to near-zero, which is a healthy sign.")


In [ ]:
# Collect FINE-TUNED answers on the exact same held-out prompts.

FastLanguageModel.for_inference(model)

ft_domain_outputs = []
print("FINE-TUNED MODEL on domain reviews (short analyzer prompt):\n")
for r in domain_reviews:
    out = generate_response(r, ANALYZER_SYSTEM)
    ft_domain_outputs.append(out)
    print("REVIEW:", short(r, 80))
    print("OUTPUT:", short(out, 160))
    print("valid JSON:", is_valid_analysis(out), "\n")

ft_general_outputs = []
print("\nFINE-TUNED MODEL on general questions (neutral prompt):\n")
for q in general_questions:
    out = generate_response(q, NEUTRAL_SYSTEM)
    ft_general_outputs.append(out)
    print("Q:", q)
    print("A:", short(out, 160), "\n")


In [ ]:
# Side-by-side comparison table and the quantitative valid-JSON rates.

print("BASE vs FINE-TUNED on held-out reviews:\n")
for r, b, f in zip(domain_reviews, base_domain_outputs, ft_domain_outputs):
    print("REVIEW    :", short(r, 90))
    print("BASE      :", short(b, 90))
    print("FINE-TUNED:", short(f, 90))
    print()

base_domain_valid = sum(is_valid_analysis(o) for o in base_domain_outputs)
ft_domain_valid = sum(is_valid_analysis(o) for o in ft_domain_outputs)
base_general_valid = sum(is_valid_analysis(o) for o in base_general_outputs)
ft_general_valid = sum(is_valid_analysis(o) for o in ft_general_outputs)

n_dom = len(domain_reviews)
n_gen = len(general_questions)
base_domain_rate = 100 * base_domain_valid / n_dom
ft_domain_rate = 100 * ft_domain_valid / n_dom
base_general_rate = 100 * base_general_valid / n_gen
ft_general_rate = 100 * ft_general_valid / n_gen

print("Valid-JSON rates:")
print(f"  Domain reviews    -> base: {base_domain_rate:.0f}%   fine-tuned: {ft_domain_rate:.0f}%")
print(f"  General questions -> base: {base_general_rate:.0f}%   fine-tuned: {ft_general_rate:.0f}%")
print("\nReading: high rate on domain reviews = success (the goal). Any rate on general questions")
print("for the fine-tuned model is the 'format bleeding' side effect worth discussing in the report.")


In [ ]:
# Figure 5: the headline result. Valid-JSON rate, base vs fine-tuned, hatched + gradient bars.

groups = ["Domain Reviews", "General Questions"]
base_rates = [base_domain_rate, base_general_rate]
ft_rates = [ft_domain_rate, ft_general_rate]
print("Figure 5 data -> base:", base_rates, "| fine-tuned:", ft_rates)

x = np.arange(len(groups))
bw = 0.36
fig, ax = plt.subplots(figsize=(9, 5.8))
b1 = ax.bar(x - bw / 2, base_rates, bw, color=lighten(PALETTE["slate"], 0.25),
            edgecolor="white", linewidth=1.6, hatch="...", zorder=3, label="Base model")
b2 = ax.bar(x + bw / 2, ft_rates, bw, color=lighten(PALETTE["green"], 0.12),
            edgecolor="white", linewidth=1.6, hatch="//", zorder=3, label="Fine-tuned")
for bars in (b1, b2):
    for b in bars:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 1.8,
                f"{b.get_height():.0f}%", ha="center", va="bottom", fontsize=13, weight="bold")
ax.set_title("Valid JSON Output Rate: Base vs Fine-tuned", pad=14, weight="bold")
ax.set_ylabel("Valid JSON Rate (%)")
ax.set_xticks(x)
ax.set_xticklabels(groups)
ax.set_ylim(0, 116)
ax.legend(frameon=False, loc="upper right")
ax.yaxis.grid(True)
tidy(ax)
fig.tight_layout()
fig.savefig(os.path.join(IMG_DIR, "05_json_rate_comparison.png"), dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# Export. Save the lightweight LoRA adapter (always works), then export to GGUF for local use
# with Ollama or llama.cpp. The GGUF step downloads the full-precision weights to merge and
# builds llama.cpp, so it needs network and a few minutes. It is wrapped so a failure here will
# not crash the notebook. If your download keeps dropping, uncomment the HF mirror line below.

# import os; os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"   # faster/steadier downloads in some regions

model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("Saved LoRA adapter to ./lora_model")

try:
    model.save_pretrained_gguf("model_gguf", tokenizer, quantization_method="q4_k_m")
    print("Saved GGUF model to ./model_gguf (quantization q4_k_m)")
except Exception as e:
    print("GGUF export skipped:", e)
    print("The LoRA adapter above is already saved; you can redo GGUF later on a stable network.")


In [1]:

import threading


_base_thread_excepthook = threading.excepthook


def _silence_thread_decode_error(args):
    if args.exc_type is not None and issubclass(args.exc_type, UnicodeDecodeError):
        return
    _base_thread_excepthook(args)


threading.excepthook = _silence_thread_decode_error

import json
import torch
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048
LORA_PATH = "lora_model"          # directory holding the saved LoRA adapter

# --------------------------------------------------------------------
# 1. Load the fine-tuned model (base weights + trained LoRA adapter).
# --------------------------------------------------------------------
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LORA_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,                   # let Unsloth pick fp16/bf16 for the GPU
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)   # enable 2x faster inference

# Qwen's generation config ships a large default max_length. Passing
# max_new_tokens alongside it triggers a "both are set" warning, so clear it
# and let the generated length be governed solely by max_new_tokens.
model.generation_config.max_length = None

print(f"Fine-tuned model loaded from ./{LORA_PATH}\n")

# --------------------------------------------------------------------
# 2. Inference and JSON-validation helpers (identical to the training run).
# --------------------------------------------------------------------
ANALYZER_SYSTEM = "You are a customer review analysis assistant."
REQUIRED_KEYS = {"sentiment", "score", "topics", "summary"}
VALID_SENTIMENTS = {"positive", "negative", "neutral", "mixed"}


def generate_response(user_msg, system_msg, max_new_tokens=160):
    messages = [{"role": "system", "content": system_msg},
                {"role": "user", "content": user_msg}]
    enc = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to("cuda")
    out = model.generate(
        **enc, max_new_tokens=max_new_tokens,
        temperature=0.7, top_p=0.8, use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = out[0][enc["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def extract_json(text):
    # Return the first balanced {...} block parsed as JSON, or None.
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(text[start:i + 1])
                except Exception:
                    return None
    return None


def is_valid_analysis(text):
    # True only if the output is a JSON object matching the target schema.
    obj = extract_json(text)
    return (isinstance(obj, dict)
            and REQUIRED_KEYS.issubset(obj.keys())
            and obj.get("sentiment") in VALID_SENTIMENTS
            and isinstance(obj.get("score"), int) and 1 <= obj["score"] <= 5
            and isinstance(obj.get("topics"), list)
            and isinstance(obj.get("summary"), str))


# --------------------------------------------------------------------
# 3. Held-out reviews for the comparison (none appear in the training set).
# --------------------------------------------------------------------
reviews = [
    "The blender works fine but the lid leaks every single time I make a smoothie. Annoying.",
    "Absolutely love this backpack. Tons of pockets, super comfy straps, and it survived a rainstorm without leaking.",
    "Battery dies in about three hours and it gets uncomfortably hot while charging. Not what I expected for the price.",
    "Fantastic noise cancelling on my flight, and the call quality blew me away. Worth every cent.",
]

# --------------------------------------------------------------------
# 4. Same model, two behaviors: adapter disabled = base, enabled = fine-tuned.
# --------------------------------------------------------------------
base_valid = ft_valid = 0

for r in reviews:
    print("=" * 90)
    print("REVIEW      :", r)

    # Adapter disabled: the model reverts to the untouched base behavior.
    with model.disable_adapter():
        base_out = generate_response(r, ANALYZER_SYSTEM)
    bv = is_valid_analysis(base_out)
    base_valid += bv
    print("\n[BASE]      ", base_out)
    print("valid JSON  :", bv)

    # Adapter enabled: the fine-tuned behavior.
    ft_out = generate_response(r, ANALYZER_SYSTEM)
    fv = is_valid_analysis(ft_out)
    ft_valid += fv
    print("\n[FINE-TUNED]", ft_out)
    print("valid JSON  :", fv)
    print()

# --------------------------------------------------------------------
# 5. Aggregate valid-JSON rate for each model.
# --------------------------------------------------------------------
n = len(reviews)
print("=" * 90)
print(f"Valid JSON rate   ->  base: {100*base_valid/n:.0f}%   "
      f"fine-tuned: {100*ft_valid/n:.0f}%   ({n} reviews)")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0621 12:53:06.054000 65668 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0621 12:53:06.095000 65668 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


<string>:1: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.


==((====))==  Unsloth 2026.6.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 5060 Ti. Num GPUs = 1. Max memory: 15.928 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.12.0+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.7.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


...

unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.6.7 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Fine-tuned model loaded from ./lora_model

REVIEW      : The blender works fine but the lid leaks every single time I make a smoothie. Annoying.


D:\Anaconda\envs\usesam3\Lib\site-packages\transformers\modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
D:\Anaconda\envs\usesam3\Lib\site-packages\transformers\modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



[BASE]       It sounds like you're experiencing a common issue with your blender. The leaking lid can be frustrating, especially when you're trying to enjoy a smoothie. Here are a few steps you might consider to address this problem:

1. **Check the Seals**: Ensure that all the seals and gaskets around the lid are properly seated and not damaged. Sometimes, they can become loose or worn out over time.

22. **Clean the Lid Properly**: Make sure the lid is clean and free of any food particles or residue that could interfere with its seal.

3. **Inspect the Lid for Damage**: Look closely at the lid and the sealing area for any cracks or damage that might be causing the leak.

4. **Replace the Lid**: If the seals or gasket are old or worn
valid JSON  : False

[FINE-TUNED] {"sentiment": "neutral", "score": 3, "topics": ["performance", "ease of cleaning"], "summary": "Customer is neutral, noting performance and ease of cleaning."}
valid JSON  : True

REVIEW      : Absolutely love this backp